# Séance 7 — Introduction à PySpark

**Durée pratique : 3 h** &nbsp;·&nbsp; Ateliers 7.1 à 7.4

## Ce que vous saurez faire à la fin

- démarrer une session Spark locale et charger des données avec un schéma explicite ;
- traduire en PySpark les opérations Pandas que vous maîtrisez déjà ;
- lire un plan d'exécution et comprendre pourquoi rien ne se passe avant l'action ;
- mesurer le point de bascule entre Pandas et Spark, et le justifier.

## Préparation obligatoire avant la séance

```bash
pip install pyspark
python src/generate_data.py --big 5000000   # environ 400 Mo, comptez 3 à 5 minutes
```

Spark exige un environnement Java (JDK 8, 11 ou 17). Vérifiez avec `java -version`.

> **Convention de nommage du module.** Le code est écrit en anglais et suit la PEP 8 :
> fonctions et variables en `snake_case`, constantes en `MAJUSCULES`. Les **noms de colonnes**
> restent en français parce qu'ils viennent de la source : renommer les colonnes d'un fichier
> d'entrée est une transformation comme une autre, elle se décide et se documente, elle ne se
> fait pas par réflexe. Vous rencontrerez cette situation partout en entreprise.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

print('Données disponibles :')
for path in sorted(RAW_DIR.glob('*')):
    print(' ', path.name)


# Parquet conserve les types (dates, entiers, catégories) là où le CSV les perd :
# c'est le format à privilégier entre deux étapes d'un pipeline. Repli automatique
# sur le CSV si pyarrow n'est pas installé.
def save_dataset(df, name):
    try:
        path = PROCESSED_DIR / f'{name}.parquet'
        df.to_parquet(path, index=False)
    except ImportError:
        path = PROCESSED_DIR / f'{name}.csv'
        df.to_csv(path, index=False)
        print('(pyarrow absent : repli sur le CSV)')
    print('écrit :', path.name, df.shape)
    return path


def load_dataset(name):
    parquet_path = PROCESSED_DIR / f'{name}.parquet'
    csv_path = PROCESSED_DIR / f'{name}.csv'
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(f'{name} introuvable : exécutez le notebook précédent')


def dataset_exists(name):
    return ((PROCESSED_DIR / f'{name}.parquet').exists()
            or (PROCESSED_DIR / f'{name}.csv').exists())

Données disponibles :
  capteurs.csv
  clients.csv
  evenements_volumineux.csv
  magasins.csv
  produits.csv
  ventes_brutes.csv
  ventes_extrait.json
  ventes_extrait.xlsx


---
## Atelier 7.1 — Session Spark et schéma explicite (40 min)

### Partie guidée

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (DoubleType, IntegerType, LongType, StringType,
                               StructField, StructType, TimestampType)

spark = (SparkSession.builder
         .appName('PMD-M1')
         .master('local[*]')                            # tous les cœurs de la machine
         .config('spark.driver.memory', '4g')
         .config('spark.sql.shuffle.partitions', '16')  # défaut 200, trop pour du local
         .getOrCreate())

spark.sparkContext.setLogLevel('ERROR')
print('Spark', spark.version)
print('Cœurs disponibles :', spark.sparkContext.defaultParallelism)
print(interface := spark.sparkContext.uiWebUrl)  # URL de l'interface web Spark


PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

**`spark.sql.shuffle.partitions` à 200 par défaut** est un réglage pensé pour un cluster.
En local il crée 200 tâches minuscules dont l'orchestration coûte plus cher que le calcul.
C'est la première cause de la réputation de lenteur de Spark sur petits volumes.

In [ ]:
import time

events_path = str(RAW_DIR / 'evenements_volumineux.csv')

# Méthode 1 : inférence automatique du schéma. Simple, mais Spark lit le fichier
# une première fois uniquement pour deviner les types.
start = time.perf_counter()
events_inferred = spark.read.csv(events_path, header=True, inferSchema=True)
print(f'Inférence : {time.perf_counter() - start:.1f} s')
events_inferred.printSchema()

In [ ]:
# Méthode 2 : schéma explicite. Une seule lecture, et aucune surprise de typage.
EVENTS_SCHEMA = StructType([
    StructField('id_evenement', LongType(), True),
    StructField('horodatage', TimestampType(), True),
    StructField('id_utilisateur', IntegerType(), True),
    StructField('categorie', StringType(), True),
    StructField('ville', StringType(), True),
    StructField('montant', DoubleType(), True),
    StructField('duree_s', IntegerType(), True),
])

start = time.perf_counter()
events = spark.read.csv(events_path, header=True, schema=EVENTS_SCHEMA)
print(f'Schéma explicite : {time.perf_counter() - start:.3f} s')

L'écart est spectaculaire, et il a une explication simple : avec un schéma explicite,
**rien n'a encore été lu**. Spark s'est contenté d'enregistrer une intention.

In [ ]:
# La preuve : count() est une ACTION, elle déclenche le calcul réel
start = time.perf_counter()
n_rows = events.count()
print(f'{n_rows} lignes comptées en {time.perf_counter() - start:.1f} s')

### Partie autonome

In [ ]:
# Q1. Charger le fichier ventes_brutes.csv dans Spark avec un schéma explicite.
#     Attention : prix_unitaire contient du texte pour une partie des lignes.
#     Quel type déclarez-vous, et pourquoi ?

# TODO

---
## Atelier 7.2 — Rejouer le nettoyage en PySpark (55 min)

### Partie guidée : le dictionnaire de traduction

| Pandas | PySpark |
|---|---|
| `df[df.a > 5]` | `df.filter(F.col('a') > 5)` |
| `df['b'] = df['a'] * 2` | `df.withColumn('b', F.col('a') * 2)` |
| `df[['a','b']]` | `df.select('a', 'b')` |
| `df.rename(columns={'a':'c'})` | `df.withColumnRenamed('a', 'c')` |
| `df.groupby('a')['b'].sum()` | `df.groupBy('a').agg(F.sum('b'))` |
| `df.drop_duplicates()` | `df.dropDuplicates()` |
| `df.dropna(subset=['a'])` | `df.na.drop(subset=['a'])` |
| `df.fillna({'a': 0})` | `df.na.fill({'a': 0})` |
| `df.sort_values('a')` | `df.orderBy('a')` |
| `df.head(5)` | `df.show(5)` ou `df.limit(5).toPandas()` |

**La différence de fond n'est pas syntaxique.** En Pandas, chaque ligne s'exécute
immédiatement. En Spark, elle s'ajoute à un plan qui ne sera évalué qu'à la première action.

In [ ]:
clean_events = (events
                .filter(F.col('montant') > 0)
                .filter(F.col('duree_s').between(1, 7200))
                .withColumn('ville', F.lower(F.trim(F.col('ville'))))
                .withColumn('jour', F.to_date('horodatage'))
                .withColumn('heure', F.hour('horodatage'))
                .withColumn('est_weekend', F.dayofweek('horodatage').isin([1, 7]))
                .withColumn('tranche_montant',
                            F.when(F.col('montant') < 30, 'faible')
                             .when(F.col('montant') < 100, 'moyen')
                             .otherwise('eleve'))
                .dropDuplicates(['id_evenement']))

print('Aucun calcul lancé à ce stade.')
clean_events.printSchema()

In [ ]:
# Le plan d'exécution : ce que Spark VA faire, et comment Catalyst l'a réorganisé
clean_events.explain(mode='formatted')

Repérez dans le plan la remontée des filtres au plus près de la lecture du fichier
(*predicate pushdown*) et la sélection des seules colonnes utiles (*column pruning*).
Vous n'avez pas demandé ces optimisations : Catalyst les a déduites de votre plan.
C'est la contrepartie de l'évaluation paresseuse.

### Partie autonome

In [ ]:
# Q2. Écrire l'équivalent PySpark du nettoyage de la séance 3 sur les ventes :
#     conversion du prix textuel, normalisation des colonnes texte, suppression des
#     doublons, parsing de la date, calcul du montant.
#     Indices : F.regexp_replace, F.to_timestamp, F.coalesce.

# TODO

**Q3.** Dans votre version PySpark, quelle opération du nettoyage Pandas avez-vous eu le
plus de mal à traduire ? Pourquoi ?

*Votre réponse :*

---
## Atelier 7.3 — Agrégations, jointures et partitions (45 min)

In [ ]:
summary = (clean_events
           .groupBy('ville', 'categorie')
           .agg(F.count('*').alias('nb_evenements'),
                F.round(F.sum('montant'), 2).alias('ca_total'),
                F.round(F.avg('montant'), 2).alias('montant_moyen'),
                F.expr('percentile_approx(montant, 0.5)').alias('montant_median'),
                F.countDistinct('id_utilisateur').alias('utilisateurs_uniques'))
           .orderBy(F.desc('ca_total')))

summary.show(10, truncate=False)

`percentile_approx` calcule une médiane **approchée**. Une médiane exacte imposerait de
trier l'intégralité des données sur l'ensemble des machines, ce qui coûte infiniment plus
cher. Sur des volumes de cet ordre, l'approximation est indiscernable du résultat exact.

In [ ]:
# Écriture partitionnée : chaque valeur de partition devient un sous-dossier
output_path = str(PROCESSED_DIR / 'evenements_spark')

(clean_events
 .select('id_evenement', 'jour', 'ville', 'categorie', 'montant', 'duree_s')
 .write.mode('overwrite')
 .partitionBy('categorie')
 .parquet(output_path))

print('Écrit dans', output_path)
for folder in sorted(os.listdir(output_path))[:6]:
    print(' ', folder)

Le partitionnement sur disque permet à une lecture filtrée sur `categorie` de ne toucher
que le sous-dossier concerné. Choisissez la clé de partitionnement en fonction des filtres
les plus fréquents, et évitez les clés de forte cardinalité : mille partitions de dix lignes
sont pires que pas de partitionnement du tout.

In [ ]:
# Q4. Reproduire en PySpark l'assemblage de la séance 6 : joindre les ventes
#     aux produits et aux magasins, puis produire le chiffre d'affaires par région
#     et par catégorie. Comparez le résultat à celui obtenu en Pandas.

# TODO

In [ ]:
# Q5. Exécuter .explain() sur votre jointure. Repérez le type de jointure choisi
#     par Spark (BroadcastHashJoin ou SortMergeJoin) et expliquez ce choix
#     au regard de la taille des tables.

# TODO

---
## Atelier 7.4 — Où se situe le point de bascule (40 min)

C'est l'atelier dont vous devrez rendre compte en soutenance. La question *pourquoi avoir
utilisé Spark ici ?* appelle une réponse chiffrée, pas une intuition.

In [ ]:
def benchmark_pandas(path, n_rows):
    start = time.perf_counter()
    df = pd.read_csv(path, nrows=n_rows)
    result = (df[df['montant'] > 0]
              .groupby(['ville', 'categorie'])
              .agg(nb=('id_evenement', 'count'), ca=('montant', 'sum')))
    duration = time.perf_counter() - start
    memory_mb = df.memory_usage(deep=True).sum() / 1024 ** 2
    del df
    return duration, memory_mb, len(result)


def benchmark_spark(path, n_rows):
    start = time.perf_counter()
    df = spark.read.csv(path, header=True, schema=EVENTS_SCHEMA).limit(n_rows)
    result = (df.filter(F.col('montant') > 0)
              .groupBy('ville', 'categorie')
              .agg(F.count('*').alias('nb'), F.sum('montant').alias('ca')))
    n_groups = result.count()
    return time.perf_counter() - start, None, n_groups

In [ ]:
SAMPLE_SIZES = [50_000, 250_000, 1_000_000, 3_000_000]
measurements = []

for n_rows in SAMPLE_SIZES:
    pandas_duration, memory_mb, _ = benchmark_pandas(events_path, n_rows)
    spark_duration, _, _ = benchmark_spark(events_path, n_rows)
    measurements.append({
        'lignes': n_rows,
        'pandas_s': round(pandas_duration, 2),
        'spark_s': round(spark_duration, 2),
        'memoire_pandas_Mo': round(memory_mb, 1),
        'rapport': round(pandas_duration / spark_duration, 2),
    })
    print(measurements[-1])

comparison = pd.DataFrame(measurements)
comparison

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(comparison['lignes'], comparison['pandas_s'], marker='o', label='pandas')
ax.plot(comparison['lignes'], comparison['spark_s'], marker='s', label='pyspark')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('nombre de lignes')
ax.set_ylabel('durée (s)')
ax.set_title('Point de bascule Pandas / PySpark')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Interprétation attendue

Sur une machine unique, **Pandas gagne presque toujours en durée brute**. Spark paie un coût
fixe de démarrage et de coordination que le parallélisme local ne compense pas.

L'intérêt de Spark n'est donc pas la vitesse à cette échelle, mais :

- la capacité à traiter un volume qui ne tient pas en mémoire ;
- la possibilité de passer du local au cluster sans réécrire le code ;
- la lecture sélective de données partitionnées, sans charger le reste.

**Répondre « j'ai utilisé Spark parce que c'est plus rapide » en soutenance est une réponse
fausse et sera sanctionnée.** La bonne réponse porte sur le volume, la mémoire disponible
et la trajectoire du projet.

In [ ]:
# Q6. Compléter le tableau : à partir de quel volume Pandas dépasse-t-il la mémoire
#     disponible sur VOTRE machine ? Estimez-le à partir de la colonne memoire_pandas_Mo
#     et de la RAM dont vous disposez. Écrivez la valeur et le raisonnement.

# TODO

In [ ]:
spark.stop()
print('Session fermée.')

---
## Livrable de la séance

- un script `src/spark_pipeline.py` autonome, exécutable par `spark-submit`, produisant
  un Parquet partitionné ;
- le tableau comparatif chiffré Pandas / PySpark et le graphique associé ;
- un paragraphe écrit justifiant, pour votre projet, le choix de traiter telle étape en
  Spark plutôt qu'en Pandas.